In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")

In [4]:
response = model.invoke("hello")
response.content

'Hello! 👋 How can I help you today?'

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at a location"""
    return f"The weather is cloudy in {location}"

model_with_tool = model.bind_tools([get_weather])

In [7]:
response = model_with_tool.invoke("What's the weather like in Chandigarh?")
print(response)
for toolCall in response.tool_calls:
    print(f"Tool: {toolCall['name']}")
    print(f"Args: {toolCall['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks for the weather in Chandigarh. We can use the get_weather function.', 'tool_calls': [{'id': 'fc_3cd15d83-ba1b-45e7-ab12-f23af3ebb8a8', 'function': {'arguments': '{"location":"Chandigarh"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 127, 'total_tokens': 170, 'completion_time': 0.045308425, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.00736896, 'prompt_tokens_details': None, 'queue_time': 0.04744787, 'total_time': 0.052677385}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e64d6-b2eb-7601-a812-515382745958-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Chandigarh'}, 'id': 'fc_3cd15d83-ba1b-45e7-ab12-f23af3ebb8a8', 'type': 'tool_call'}] invalid_tool_calls=[] 

In [8]:
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tool.invoke(messages)
print(final_response.text)

The weather is cloudy in Boston.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the get_weather function.', 'tool_calls': [{'id': 'fc_e9f8067d-a053-455f-a571-387551146ed4', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 126, 'total_tokens': 159, 'completion_time': 0.033664206, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.0070649, 'prompt_tokens_details': None, 'queue_time': 0.05112763, 'total_time': 0.040729106}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e64d8-80b0-7e82-b1f6-3a71c36fa3f7-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_e9f8067d-a053-455f-a571-387551146ed4', 'type': 'tool_ca